In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA_DIR = Path("../group_7/data/sleep/") 
FMT = "%m/%d/%Y %I:%M:%S %p"

daily = pd.read_csv(DATA_DIR / "sleepDay_merged.csv").drop_duplicates()
daily["Id"] = daily["Id"].astype(str)
daily["sleep_day"] = pd.to_datetime(daily["SleepDay"], format=FMT).dt.normalize()

minute = pd.read_csv(DATA_DIR / "minuteSleep_merged.csv").drop_duplicates()
minute["Id"] = minute["Id"].astype(str)
minute["date"] = pd.to_datetime(minute["date"], format=FMT)

print(daily.shape, minute.shape)

(410, 6) (187978, 4)


In [2]:
STATE = {1: "Asleep", 2: "Restless", 3: "Awake"}
STATES = list(STATE.values())
minute["state"] = minute["value"].map(STATE)


nights = daily.rename(columns={
    "TotalMinutesAsleep": "min_asleep",
    "TotalTimeInBed": "min_in_bed",
    "TotalSleepRecords": "n_records",
})[["Id", "sleep_day", "n_records", "min_asleep", "min_in_bed"]]

nights["hours_asleep"] = nights["min_asleep"] / 60
nights["efficiency"] = 100 * nights["min_asleep"] / nights["min_in_bed"]
nights["awake_in_bed"] = nights["min_in_bed"] - nights["min_asleep"]
nights["weekday"] = nights["sleep_day"].dt.day_name()
nights["is_weekend"] = nights["sleep_day"].dt.dayofweek >= 5

counts = (minute.groupby(["Id", "logId", "state"]).size()
          .unstack(fill_value=0).reindex(columns=STATES, fill_value=0))
span = minute.groupby(["Id", "logId"])["date"].agg(start="min", end="max")
sessions = span.join(counts).reset_index()
sessions["minutes"] = sessions[STATES].sum(axis=1)
sessions["sleep_day"] = sessions["end"].dt.normalize()

def bed_clock(ts):
    """Hours after midnight, shifted so 1 am = 25 (keeps 11 pm and 1 am next to each other)."""
    h = ts.dt.hour + ts.dt.minute / 60
    return h.where(h >= 12, h + 24)

sessions["bed_hour"] = bed_clock(sessions["start"])
sessions["wake_hour"] = sessions["end"].dt.hour + sessions["end"].dt.minute / 60

# ignores naps
main = (sessions.sort_values("minutes", ascending=False)
        .drop_duplicates(["Id", "sleep_day"]))


nights = nights.merge(
    main[["Id", "sleep_day", "start", "end", "bed_hour", "wake_hour"]],
    on=["Id", "sleep_day"], how="left")

nights.head()

,Id,sleep_day,n_records,min_asleep,min_in_bed,hours_asleep,efficiency,awake_in_bed,weekday,is_weekend,start,end,bed_hour,wake_hour
0,1503960366,2016-04-12,1,327,346,5.450000,94.508671,19,Tuesday,False,2016-04-12 02:47:30,2016-04-12 08:32:30,26.783333,8.533333
1,1503960366,2016-04-13,2,384,407,6.400000,94.348894,23,Wednesday,False,2016-04-13 03:08:30,2016-04-13 08:20:30,27.133333,8.333333
2,1503960366,2016-04-15,1,412,442,6.866667,93.212670,30,Friday,False,2016-04-15 02:59:00,2016-04-15 10:20:00,26.983333,10.333333
3,1503960366,2016-04-16,2,340,367,5.666667,92.643052,27,Saturday,True,2016-04-16 02:11:00,2016-04-16 06:59:00,26.183333,6.983333
4,1503960366,2016-04-17,1,700,712,11.666667,98.314607,12,Sunday,True,2016-04-16 23:27:00,2016-04-17 11:18:00,23.450000,11.300000


In [3]:
WEEK_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
COL = {
    "main": "#4C78A8", "accent": "#F58518", "good": "#54A24B", "grey": "#B8B8B8",
    "Asleep": "#4C78A8", "Restless": "#F2CF5B", "Awake": "#E45756",
}

def _style(fig, title, height=380):
    fig.update_layout(
        title=dict(text=title, x=0.02),
        template="plotly_white",
        height=height,
        margin=dict(l=50, r=20, t=60, b=40),
        font=dict(size=13),
    )
    return fig

def fig_hours_hist(df):
    fig = px.histogram(df, x="hours_asleep", nbins=30, color_discrete_sequence=[COL["main"]])
    fig.add_vrect(x0=7, x1=9, fillcolor=COL["good"], opacity=0.12, line_width=0,
                  annotation_text="7-9 h", annotation_position="top left")
    fig.update_xaxes(title="Hours asleep")
    fig.update_yaxes(title="Nights")
    return _style(fig, "How long do people sleep?")


def fig_weekday(df):
    g = (df.groupby("weekday")
           .agg(hours=("hours_asleep", "mean"), n=("hours_asleep", "size"))
           .reindex(WEEK_ORDER).dropna().reset_index())
    fig = px.bar(g, x="weekday", y="hours", custom_data=["n"], text="hours",
                 color_discrete_sequence=[COL["main"]])
    fig.update_traces(
        texttemplate="%{y:.1f}",
        hovertemplate="%{x}<br>%{y:.2f} h average<br>%{customdata[0]} nights<extra></extra>")
    fig.update_xaxes(title=None)
    fig.update_yaxes(title="Mean hours asleep", rangemode="tozero")
    return _style(fig, "Sleep by day of week")


In [4]:
fig_hours_hist(nights).show()


In [5]:
fig_weekday(nights).show()